### Импорт необходимых библиотек

In [15]:
import pandas as pd
import numpy as np
import gdown
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error
from lightgbm import LGBMRegressor
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')
from scipy.stats import boxcox
from sklearn.preprocessing import PowerTransformer

### Загрузка данных и первичный анализ

In [16]:
# ID файлов
train_id = '159PZX3X5rpUO-WbzWyC9whnc8B4mNqJl'
test_id = '1Ui2t87X3in-Wu-pnjkDXa_VtPsVafi0l'
sample_id = '1LL6moSzpUVxJUTMeXihWvUxBJNjvj6EH'

# Скачиваем файлы
gdown.download(f'https://drive.google.com/uc?id={train_id}', 'train.csv', quiet=False)
gdown.download(f'https://drive.google.com/uc?id={test_id}', 'test.csv', quiet=False)
gdown.download(f'https://drive.google.com/uc?id={sample_id}', 'sample_submission.csv', quiet=False)

# Читаем файлы
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
df_sample = pd.read_csv('sample_submission.csv')

# Выводим информацию
print(f"\nTrain shape: {df_train.shape}")
print(f"\nTest shape: {df_test.shape}")
print(f"\nSample shape: {df_sample.shape}")

print("\nTrain head")
print(df_train.head())

print("\nTest head")
print(df_test.head())

print("\nSample head")
print(df_sample.head())

print("\nTRAIN describe")
print(df_train.describe())

print("\nTEST describe")
print(df_test.describe())

# Проверка пропусков
train_nulls = df_train.isnull().sum().sum()
test_nulls = df_test.isnull().sum().sum()

print(f"\nПропуски в train: {train_nulls}")
print(f"\nПропуски в test: {test_nulls}")

Downloading...
From: https://drive.google.com/uc?id=159PZX3X5rpUO-WbzWyC9whnc8B4mNqJl
To: /content/train.csv
100%|██████████| 1.36M/1.36M [00:00<00:00, 148MB/s]
Downloading...
From: https://drive.google.com/uc?id=1Ui2t87X3in-Wu-pnjkDXa_VtPsVafi0l
To: /content/test.csv
100%|██████████| 441k/441k [00:00<00:00, 102MB/s]
Downloading...
From: https://drive.google.com/uc?id=1LL6moSzpUVxJUTMeXihWvUxBJNjvj6EH
To: /content/sample_submission.csv
100%|██████████| 15.4k/15.4k [00:00<00:00, 34.9MB/s]



Train shape: (751, 214)

Test shape: (250, 211)

Sample shape: (250, 4)

Train head
   index    IC50, mM    CC50, mM          SI  MaxAbsEStateIndex  \
0      0  102.414420   95.757483    0.935000           5.466584   
1      1    0.044333    8.401080  189.500000          11.492712   
2      2    4.437964   50.085589   11.285714           5.366084   
3      3    6.827881  682.788051  100.000000          13.317130   
4      4    2.003253   70.001455   34.943894           6.320833   

   MaxEStateIndex  MinAbsEStateIndex  MinEStateIndex       qed        SPS  \
0        5.466584           0.719259        0.719259  0.681165  18.307692   
1       11.492712           0.012350       -3.798024  0.769122  27.652174   
2        5.366084           0.522930        0.522930  0.612606  24.608696   
3       13.317130           0.020658       -4.829339  0.345823  12.400000   
4        6.320833           0.300347        0.300347  0.562066  60.272727   

   ...  fr_sulfide  fr_sulfonamd  fr_sulfone  fr_

Данные успешно загружены: обучающая выборка содержит 751 вещество с 211 химическими дескрипторами и тремя целевыми переменными (IC50, CC50, SI), тестовая выборка — 250 веществ с теми же 211 признаками без целевых переменных, файл sample_submission задаёт формат для предсказаний.  Распределения признаков в train и test сопоставимы, поскольку данные были случайным образом перемешаны перед разделением на выборки, что подтверждается близостью средних значений и стандартных отклонений в describe (например, для MaxAbsEStateIndex: train mean=10.86, test mean=10.75). В данных присутствуют пропуски: 24 в обучающей выборке и 12 в тестовой, которые потребуется обработать. Целевые переменные имеют высокую вариативность из-за того, что вещества сильно различаются по биологической активности: IC50 варьируется от 0.0035 mM  до 4095 mM — различие более чем в миллион раз, аналогично CC50 (от 0.7 до 4539 mM) и SI (от 0.01 до 15620).

### Обработка пропусков

In [17]:
# Какие колонки имеют пропуски и сколько
print("\nПропуски в train")
train_nulls = df_train.isnull().sum()
train_nulls = train_nulls[train_nulls > 0].sort_values(ascending=False)
print(train_nulls)

print("\nПропуски в test")
test_nulls = df_test.isnull().sum()
test_nulls = test_nulls[test_nulls > 0].sort_values(ascending=False)
print(test_nulls)



Пропуски в train
MaxPartialCharge       2
MinPartialCharge       2
MaxAbsPartialCharge    2
MinAbsPartialCharge    2
BCUT2D_MWHI            2
BCUT2D_MWLOW           2
BCUT2D_CHGHI           2
BCUT2D_CHGLO           2
BCUT2D_LOGPHI          2
BCUT2D_LOGPLOW         2
BCUT2D_MRHI            2
BCUT2D_MRLOW           2
dtype: int64

Пропуски в test
MaxPartialCharge       1
MinPartialCharge       1
MaxAbsPartialCharge    1
MinAbsPartialCharge    1
BCUT2D_MWHI            1
BCUT2D_MWLOW           1
BCUT2D_CHGHI           1
BCUT2D_CHGLO           1
BCUT2D_LOGPHI          1
BCUT2D_LOGPLOW         1
BCUT2D_MRHI            1
BCUT2D_MRLOW           1
dtype: int64


В данных обнаружены пропуски в 12 колонках химических дескрипторов: по 2 пропуска в обучающей выборке и по 1 пропуску в тестовой выборке для каждой из этих колонок. Поскольку процент пропусков крайне мал (менее 0.3%), эти колонки удалять нецелесообразно. Применяем стратегию заполнения пропусков медианным значением, вычисленным исключительно на обучающей выборке, чтобы избежать утечки данных из тестовой выборки в процесс обучения модели.

In [18]:
# Список колонок с пропусками
null_cols = ['MaxPartialCharge', 'MinPartialCharge', 'MaxAbsPartialCharge',
             'MinAbsPartialCharge', 'BCUT2D_MWHI', 'BCUT2D_MWLOW',
             'BCUT2D_CHGHI', 'BCUT2D_CHGLO', 'BCUT2D_LOGPHI',
             'BCUT2D_LOGPLOW', 'BCUT2D_MRHI', 'BCUT2D_MRLOW']

# Заполняем пропуски медианой только из train
for col in null_cols:
    train_median = df_train[col].median()  # вычисляем на train
    df_train[col] = df_train[col].fillna(train_median)  # заполняем train
    df_test[col] = df_test[col].fillna(train_median)    # заполняем test

# Проверка
print(f"Пропуски в train: {df_train.isnull().sum().sum()}")
print(f"Пропуски в test: {df_test.isnull().sum().sum()}")

Пропуски в train: 0
Пропуски в test: 0


Для удобства работы, перебора различных вариантов подготовки данных и различных моделей напишем пайплайн

### Пайплайн

In [19]:
# Разбивка обучающей выборки на обучающую и валидационную

def split_data(df_train, target_col, test_size=0.2, random_state=42):
    """
    Разбивает данные на обучающую и валидационную выборки

    Возвращает:
        X_train, X_val, y_train, y_val, feature_cols
    """
    feature_cols = [col for col in df_train.columns
                    if col not in ['index', 'IC50, mM', 'CC50, mM', 'SI']]

    X = df_train[feature_cols]
    y = df_train[target_col]

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    return X_train, X_val, y_train, y_val, feature_cols


# Логарифмирование целевых переменных и обратное преобразование

def log_transform(y):
    return np.log1p(y)

def exp_transform(y_log):
    return np.expm1(y_log)


# Масштабирование признаков

def scale_features(X_train, X_val, X_test=None, scaler_type='standard'):
    if scaler_type == 'standard':
        scaler = StandardScaler()
    elif scaler_type == 'robust':
        scaler = RobustScaler()
    else:
        scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    result = (X_train_scaled, X_val_scaled, scaler)

    if X_test is not None:
        X_test_scaled = scaler.transform(X_test)
        result = (X_train_scaled, X_val_scaled, X_test_scaled, scaler)

    return result


# Обучение и оценка модели только на валидации

def train_and_evaluate(model, X_train, X_val, y_train, y_val, use_log=True, use_scaling=False, scaler_type='standard'):
    """
    Обучает модель на train, оценивает на val

    Возвращает:
        rmse, model
    """

    if use_log:
        y_train_transformed = log_transform(y_train)
        y_val_transformed = log_transform(y_val)
    else:
        y_train_transformed = y_train
        y_val_transformed = y_val

    if use_scaling:
        X_train_processed, X_val_processed, _ = scale_features(X_train, X_val)
    else:
        X_train_processed = X_train
        X_val_processed = X_val

    model.fit(X_train_processed, y_train_transformed)

    pred_val_transformed = model.predict(X_val_processed)

    if use_log:
        pred_val = exp_transform(pred_val_transformed)
        y_val_original = y_val
    else:
        pred_val = pred_val_transformed
        y_val_original = y_val

    rmse = np.sqrt(mean_squared_error(y_val_original, pred_val))

    return rmse, model


# Подбор гиперпараметров


def grid_search_rmse(model_class, param_grid, X_train, X_val, y_train, y_val, use_log=True, use_scaling=False, scaler_type='standard'):
    keys = param_grid.keys()
    values = param_grid.values()
    combinations = list(product(*values))

    best_rmse = float('inf')
    best_params = None
    results = []

    for combo in combinations:
        params = dict(zip(keys, combo))
        model = model_class(**params, random_state=42)

        rmse, _ = train_and_evaluate(
            model, X_train, X_val, y_train, y_val,
            use_log=use_log, use_scaling=use_scaling, scaler_type=scaler_type
        )

        results.append((params, rmse))

        if rmse < best_rmse:
            best_rmse = rmse
            best_params = params

        print(f"params: {params} -> rmse: {rmse:.4f}")

    return best_params, best_rmse, results


# Обучение на всех данных для финального сабмита


def train_final_and_predict(model, df_train, df_test, target_col, use_log=True, use_scaling=False):
    """
    Обучает модель на всех обучающих данных и предсказывает для test
    """
    feature_cols = [col for col in df_train.columns
                    if col not in ['index', 'IC50, mM', 'CC50, mM', 'SI']]

    X_full = df_train[feature_cols].copy()
    y_full = df_train[target_col].copy()
    X_test = df_test[feature_cols].copy()

    if use_log:
        y_full_transformed = log_transform(y_full)
    else:
        y_full_transformed = y_full

    if use_scaling:
        scaler = StandardScaler()
        X_full_scaled = scaler.fit_transform(X_full)
        X_test_scaled = scaler.transform(X_test)
        X_full_processed = X_full_scaled
        X_test_processed = X_test_scaled
    else:
        X_full_processed = X_full
        X_test_processed = X_test

    model.fit(X_full_processed, y_full_transformed)

    pred_test_transformed = model.predict(X_test_processed)

    if use_log:
        predictions = exp_transform(pred_test_transformed)
    else:
        predictions = pred_test_transformed

    return predictions

### Обучение моделей на валидационной выборке

In [20]:
targets = ['IC50, mM', 'CC50, mM', 'SI']

### Линейная регрессия

Простая линейная модель, которая предполагает линейную связь между признаками и целевой переменной. L2-регуляризация помогает бороться с переобучением, штрафуя большие коэффициенты. Применим для задачи как бейзлайн, чтобы понять, насколько данные линейно разделимы. Скейлинг обязателен, так как линейные модели используют расстояния и величины коэффициентов напрямую, и признаки с разными масштабами будут иметь неравномерное влияние.

In [21]:
param_grid_ridge = {'alpha': [0.1, 1.0, 10.0, 100.0]}
results_ridge = {}

for target in targets:
    print(f"\n{target}")
    X_train, X_val, y_train, y_val, _ = split_data(df_train, target)
    best_params, best_rmse, _ = grid_search_rmse(Ridge, param_grid_ridge, X_train, X_val, y_train, y_val, use_log=True, use_scaling=True)
    results_ridge[target] = best_rmse
    print(f"лучший rmse: {best_rmse:.4f}")

avg_ridge = np.mean(list(results_ridge.values()))
print(f"\nсредний rmse ridge: {avg_ridge:.4f}")


IC50, mM
params: {'alpha': 0.1} -> rmse: 644449.5625
params: {'alpha': 1.0} -> rmse: 239921.5557
params: {'alpha': 10.0} -> rmse: 5416.0511
params: {'alpha': 100.0} -> rmse: 797103789.6252
лучший rmse: 5416.0511

CC50, mM
params: {'alpha': 0.1} -> rmse: 7358.6614
params: {'alpha': 1.0} -> rmse: 3088.1094
params: {'alpha': 10.0} -> rmse: 267609447.1674
params: {'alpha': 100.0} -> rmse: 3944950059683.0469
лучший rmse: 3088.1094

SI
params: {'alpha': 0.1} -> rmse: 185651.7925
params: {'alpha': 1.0} -> rmse: 307.8086
params: {'alpha': 10.0} -> rmse: 224.9971
params: {'alpha': 100.0} -> rmse: 170.5846
лучший rmse: 170.5846

средний rmse ridge: 2891.5817


### Random Forest

Ансамбль случайных деревьев решений, усредняющий их предсказания. Устойчив к выбросам, не требует масштабирования признаков, хорошо работает с табличными данными и высокой размерностью. Применим для задачи, так как химические дескрипторы имеют сложные нелинейные зависимости, которые деревья могут улавливать. Скейлинг не требуется, так как деревья решений работают с порядком и сравнениями значений признаков, а не с расстояниями.

In [22]:
param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}
results_rf = {}

for target in targets:
    print(f"\n{target}")
    X_train, X_val, y_train, y_val, _ = split_data(df_train, target)
    best_params, best_rmse, _ = grid_search_rmse(RandomForestRegressor, param_grid_rf, X_train, X_val, y_train, y_val, use_log=True, use_scaling=False)
    results_rf[target] = best_rmse
    print(f"лучший rmse: {best_rmse:.4f}")

avg_rf = np.mean(list(results_rf.values()))
print(f"\nсредний rmse random forest: {avg_rf:.4f}")


IC50, mM
params: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 2} -> rmse: 429.6125
params: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 5} -> rmse: 429.6871
params: {'n_estimators': 100, 'max_depth': 20, 'min_samples_split': 2} -> rmse: 428.7087
params: {'n_estimators': 100, 'max_depth': 20, 'min_samples_split': 5} -> rmse: 429.9790
params: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2} -> rmse: 429.3208
params: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 5} -> rmse: 429.9291
params: {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 2} -> rmse: 428.7892
params: {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 5} -> rmse: 429.1269
params: {'n_estimators': 200, 'max_depth': 20, 'min_samples_split': 2} -> rmse: 428.1221
params: {'n_estimators': 200, 'max_depth': 20, 'min_samples_split': 5} -> rmse: 428.8130
params: {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 2} -> rmse: 428.3677
param

### XGBoost

Градиентный бустинг на деревьях, который последовательно улучшает ошибки предыдущих моделей. Имеет встроенную регуляризацию, эффективен на табличных данных. Применим для задачи, так как обычно показывает высокое качество на химических данных за счёт работы с нелинейностями и взаимодействиями признаков. Скейлинг не требуется по той же причине — деревья решений инвариантны к масштабу признаков.

In [23]:
param_grid_xgb = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6, 10],
    'learning_rate': [0.01, 0.05, 0.1]
}
results_xgb = {}

for target in targets:
    print(f"\n{target}")
    X_train, X_val, y_train, y_val, _ = split_data(df_train, target)
    best_params, best_rmse, _ = grid_search_rmse(XGBRegressor, param_grid_xgb, X_train, X_val, y_train, y_val, use_log=True, use_scaling=False)
    results_xgb[target] = best_rmse
    print(f"лучший rmse: {best_rmse:.4f}")

avg_xgb = np.mean(list(results_xgb.values()))
print(f"\nсредний rmse xgboost: {avg_xgb:.4f}")


IC50, mM
params: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.01} -> rmse: 451.7762
params: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.05} -> rmse: 431.4821
params: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1} -> rmse: 425.8010
params: {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.01} -> rmse: 445.9751
params: {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.05} -> rmse: 431.2161
params: {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.1} -> rmse: 433.6006
params: {'n_estimators': 100, 'max_depth': 10, 'learning_rate': 0.01} -> rmse: 437.5469
params: {'n_estimators': 100, 'max_depth': 10, 'learning_rate': 0.05} -> rmse: 422.2763
params: {'n_estimators': 100, 'max_depth': 10, 'learning_rate': 0.1} -> rmse: 425.9833
params: {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.01} -> rmse: 443.4303
params: {'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05} -> rmse: 427.6162
params: {'n_estimators': 200, 'ma

LightGBM

Градиентный бустинг с оптимизациями для скорости и памяти (листовой рост дерева). Работает быстрее XGBoost на больших данных, хорошо справляется с признаками, содержащими выбросы. Скейлинг не требуется, так как основан на деревьях решений.

In [24]:
param_grid_lgb = {
    'num_leaves': [15, 31, 63],
    'min_child_samples': [5, 10, 20],
    'learning_rate': [0.05, 0.1]
}
results_lgb = {}

for target in targets:
    print(f"\n{target}")
    X_train, X_val, y_train, y_val, _ = split_data(df_train, target)
    best_params, best_rmse, _ = grid_search_rmse(LGBMRegressor, param_grid_lgb, X_train, X_val, y_train, y_val, use_log=True, use_scaling=False)
    results_lgb[target] = best_rmse
    print(f"лучший rmse: {best_rmse:.4f}")

avg_lgb = np.mean(list(results_lgb.values()))
print(f"\nсредний rmse lightgbm: {avg_lgb:.4f}")


IC50, mM
params: {'num_leaves': 15, 'min_child_samples': 5, 'learning_rate': 0.05} -> rmse: 424.1196
params: {'num_leaves': 15, 'min_child_samples': 5, 'learning_rate': 0.1} -> rmse: 418.2403
params: {'num_leaves': 15, 'min_child_samples': 10, 'learning_rate': 0.05} -> rmse: 424.8115
params: {'num_leaves': 15, 'min_child_samples': 10, 'learning_rate': 0.1} -> rmse: 418.4495
params: {'num_leaves': 15, 'min_child_samples': 20, 'learning_rate': 0.05} -> rmse: 421.8269
params: {'num_leaves': 15, 'min_child_samples': 20, 'learning_rate': 0.1} -> rmse: 420.8809
params: {'num_leaves': 31, 'min_child_samples': 5, 'learning_rate': 0.05} -> rmse: 429.9578
params: {'num_leaves': 31, 'min_child_samples': 5, 'learning_rate': 0.1} -> rmse: 437.0511
params: {'num_leaves': 31, 'min_child_samples': 10, 'learning_rate': 0.05} -> rmse: 414.9954
params: {'num_leaves': 31, 'min_child_samples': 10, 'learning_rate': 0.1} -> rmse: 423.1716
params: {'num_leaves': 31, 'min_child_samples': 20, 'learning_rate': 

- Вывод по результатам моделей

- Ridge (линейная регрессия) показал очень высокий средний RMSE (2891.58). Огромные значения ошибок при некоторых alpha указывают на численную нестабильность, несмотря на масштабирование. Это подтверждает, что связь между химическими дескрипторами и целевыми переменными глубоко нелинейна, и линейная модель принципиально не подходит для задачи. Дальнейшие эксперименты с Ridge бесполезны.

- Random Forest дал средний RMSE 365.20. Лучшие параметры: n_estimators=200, max_depth=20, min_samples_split=2. Увеличение глубины деревьев до 20 и количества деревьев до 200 улучшает качество, но дальнейшее усложнение (глубина выше 20) не даёт прироста из-за ограниченности данных (600 образцов в train). Random Forest стабилен, но уступает бустингам.

- XGBoost показал средний RMSE 363.38, что немного лучше Random Forest. Лучшие параметры: для IC50 — max_depth=10, learning_rate=0.05; для CC50 — max_depth=6, learning_rate=0.05; для SI — max_depth=10, learning_rate=0.01, n_estimators=200. Результат ожидаем — бустинг обычно превосходит случайный лес, но отрыв небольшой из-за чувствительности XGBoost к выбросам, которых много в химических данных.

- LightGBM показал лучший результат среди всех моделей — средний RMSE 346.25. Лучшие параметры: для IC50 — num_leaves=31, min_child_samples=20, learning_rate=0.05; для CC50 — num_leaves=15, min_child_samples=10, learning_rate=0.1; для SI — num_leaves=31, min_child_samples=20, learning_rate=0.05. Эти параметры дают хороший баланс между сложностью модели и обобщающей способностью.

### Улучшение результатов

Эксперименты показали, что подбор гиперпараметров для существующих моделей не даёт значимого прироста качества из-за ограниченного объёма выборки (751 образец). Основной резерв повышения качества заключается в улучшении подготовки признаков. Действовать будем по следующему плану:

Остановимся на LightGBM как на лучшей модели (средний RMSE 346.25), так как он показал наилучший результат среди всех протестированных алгоритмов благодаря высокой скорости работы, устойчивости к выбросам и эффективности на табличных данных с химическими дескрипторами.

1. Проверим влияние масштабирования признаков. Хотя деревья решений теоретически не требуют нормализации, на практике StandardScaler или RobustScaler (устойчивый к выбросам) иногда дают небольшое улучшение за счёт более стабильного градиентного спуска в бустинге.

2. Проведём отбор признаков. Из 211 дескрипторов многие могут быть шумовыми или сильно коррелированными. Используем встроенную важность признаков из LightGBM для удаления наименее значимых (например, оставим топ-50 или топ-100).

3. Создадим новые признаки. Химические дескрипторы можно комбинировать: добавлять отношения, произведения, разности пар признаков, что может выявить скрытые нелинейные зависимости.

4. Обработаем выбросы в целевых переменных. IC50 и CC50 имеют экстремальные значения (до 4500), которые могут искажать обучение. Применим винсоризацию — ограничим квантилями 1-99% или удалим образцы с аномально высокими значениями.

5. Попробуем ансамблирование. Усреднение предсказаний LightGBM, XGBoost и Random Forest может снизить дисперсию ошибки и дать небольшое улучшение финального результата.

Каждый шаг будем оценивать на валидационной выборке с фиксацией среднего RMSE по трём целевым переменным.



### 1. Используем масштабирование признаков

In [25]:
param_grid_lgb = {
    'num_leaves': [15, 31, 63],
    'min_child_samples': [5, 10, 20],
    'learning_rate': [0.05, 0.1]
}

print("lightgbm с standardscaler")
results_lgb_standard = {}
for target in targets:
    print(f"\n{target}")
    X_train, X_val, y_train, y_val, _ = split_data(df_train, target)
    best_params, best_rmse, _ = grid_search_rmse(LGBMRegressor, param_grid_lgb, X_train, X_val, y_train, y_val, use_log=True, use_scaling=True, scaler_type='standard')
    results_lgb_standard[target] = best_rmse
    print(f"лучший rmse: {best_rmse:.4f}")
avg_lgb_standard = np.mean(list(results_lgb_standard.values()))
print(f"\nсредний rmse lightgbm с standardscaler: {avg_lgb_standard:.4f}")

print("\nlightgbm с robustscaler")
results_lgb_robust = {}
for target in targets:
    print(f"\n{target}")
    X_train, X_val, y_train, y_val, _ = split_data(df_train, target)
    best_params, best_rmse, _ = grid_search_rmse(LGBMRegressor, param_grid_lgb, X_train, X_val, y_train, y_val, use_log=True, use_scaling=True, scaler_type='robust')
    results_lgb_robust[target] = best_rmse
    print(f"лучший rmse: {best_rmse:.4f}")
avg_lgb_robust = np.mean(list(results_lgb_robust.values()))
print(f"\nсредний rmse lightgbm с robustscaler: {avg_lgb_robust:.4f}")


lightgbm с standardscaler

IC50, mM
params: {'num_leaves': 15, 'min_child_samples': 5, 'learning_rate': 0.05} -> rmse: 421.9667
params: {'num_leaves': 15, 'min_child_samples': 5, 'learning_rate': 0.1} -> rmse: 424.5221
params: {'num_leaves': 15, 'min_child_samples': 10, 'learning_rate': 0.05} -> rmse: 425.4953
params: {'num_leaves': 15, 'min_child_samples': 10, 'learning_rate': 0.1} -> rmse: 415.1100
params: {'num_leaves': 15, 'min_child_samples': 20, 'learning_rate': 0.05} -> rmse: 422.9549
params: {'num_leaves': 15, 'min_child_samples': 20, 'learning_rate': 0.1} -> rmse: 410.6513
params: {'num_leaves': 31, 'min_child_samples': 5, 'learning_rate': 0.05} -> rmse: 433.2827
params: {'num_leaves': 31, 'min_child_samples': 5, 'learning_rate': 0.1} -> rmse: 437.9804
params: {'num_leaves': 31, 'min_child_samples': 10, 'learning_rate': 0.05} -> rmse: 416.8313
params: {'num_leaves': 31, 'min_child_samples': 10, 'learning_rate': 0.1} -> rmse: 415.6275
params: {'num_leaves': 31, 'min_child_sampl

Масштабирование признаков дало небольшое улучшение — средний RMSE снизился с 346.25 до 343.68 (улучшение на 2.57 пункта).

StandardScaler и RobustScaler показали одинаковый результат (343.68). Это ожидаемо, так как для деревьев решений масштабирование не критично, но небольшое улучшение может быть связано с более стабильной работой оптимизационных процессов внутри бустинга.

Лучшие гиперпараметры при масштабировании изменились:

IC50: без масштабирования лучшие параметры были num_leaves=31, min_child_samples=20, learning_rate=0.05. При масштабировании лучший RMSE (397.79) достигается при нескольких комбинациях: num_leaves=31, min_child_samples=20, learning_rate=0.05 и num_leaves=63, min_child_samples=20, learning_rate=0.05 дают одинаковый результат.

CC50: без масштабирования лучшие параметры были num_leaves=15, min_child_samples=10, learning_rate=0.1 (RMSE 467.77). При масштабировании эта же комбинация остаётся лучшей.

SI: без масштабирования лучшие параметры были num_leaves=31, min_child_samples=20, learning_rate=0.05 (RMSE 165.47). При масштабировании также лучший результат 165.47 достигается при num_leaves=15/31/63, min_child_samples=20, learning_rate=0.05.


### 2. Отбор признаков

In [26]:
param_grid_lgb = {
    'num_leaves': [15, 31, 63],
    'min_child_samples': [5, 10, 20],
    'learning_rate': [0.05, 0.1]
}

best_params_top100 = {}
best_params_top50 = {}
results_top100 = {}
results_top50 = {}

for target in targets:

    print(f"цель: {target}")


    X_train, X_val, y_train, y_val, feature_cols = split_data(df_train, target)

    # обучаем модель на всех признаках
    temp_model = LGBMRegressor(num_leaves=31, min_child_samples=20, learning_rate=0.05, random_state=42, verbosity=-1)
    _, temp_fitted = train_and_evaluate(
        temp_model, X_train, X_val, y_train, y_val,
        use_log=True, use_scaling=True, scaler_type='robust'
    )

    # получаем важность признаков
    importance = temp_fitted.feature_importances_
    importance_dict = dict(zip(feature_cols, importance))
    sorted_importance = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)

    # топ-100 признаков
    top100_cols = [col for col, imp in sorted_importance[:100]]
    X_train_top100 = X_train[top100_cols]
    X_val_top100 = X_val[top100_cols]

    print("\nтоп-100 признаков:")
    best_params, best_rmse, _ = grid_search_rmse(
        LGBMRegressor, param_grid_lgb, X_train_top100, X_val_top100, y_train, y_val,
        use_log=True, use_scaling=True, scaler_type='robust'
    )
    best_params_top100[target] = best_params
    results_top100[target] = best_rmse
    print(f"\nлучший rmse: {best_rmse:.4f}")
    print(f"\nлучшие параметры: {best_params}")

    # топ-50 признаков
    top50_cols = [col for col, imp in sorted_importance[:50]]
    X_train_top50 = X_train[top50_cols]
    X_val_top50 = X_val[top50_cols]

    print("\nтоп-50 признаков:")
    best_params, best_rmse, _ = grid_search_rmse(
        LGBMRegressor, param_grid_lgb, X_train_top50, X_val_top50, y_train, y_val,
        use_log=True, use_scaling=True, scaler_type='robust'
    )
    best_params_top50[target] = best_params
    results_top50[target] = best_rmse
    print(f"\nлучший rmse: {best_rmse:.4f}")
    print(f"\nлучшие параметры: {best_params}")

# средние RMSE
avg_top100 = np.mean(list(results_top100.values()))
avg_top50 = np.mean(list(results_top50.values()))


print("\nсравнение отбора признаков")

print(f"\nтоп-100 признаков: {avg_top100:.4f}")
print(f"\nтоп-50 признаков: {avg_top50:.4f}")



цель: IC50, mM

топ-100 признаков:
params: {'num_leaves': 15, 'min_child_samples': 5, 'learning_rate': 0.05} -> rmse: 424.6727
params: {'num_leaves': 15, 'min_child_samples': 5, 'learning_rate': 0.1} -> rmse: 411.7346
params: {'num_leaves': 15, 'min_child_samples': 10, 'learning_rate': 0.05} -> rmse: 422.8201
params: {'num_leaves': 15, 'min_child_samples': 10, 'learning_rate': 0.1} -> rmse: 417.2023
params: {'num_leaves': 15, 'min_child_samples': 20, 'learning_rate': 0.05} -> rmse: 424.5417
params: {'num_leaves': 15, 'min_child_samples': 20, 'learning_rate': 0.1} -> rmse: 424.3967
params: {'num_leaves': 31, 'min_child_samples': 5, 'learning_rate': 0.05} -> rmse: 430.8994
params: {'num_leaves': 31, 'min_child_samples': 5, 'learning_rate': 0.1} -> rmse: 435.0102
params: {'num_leaves': 31, 'min_child_samples': 10, 'learning_rate': 0.05} -> rmse: 409.4617
params: {'num_leaves': 31, 'min_child_samples': 10, 'learning_rate': 0.1} -> rmse: 424.0997
params: {'num_leaves': 31, 'min_child_sample

Отбор признаков не дал значимого улучшения. Разница между топ-100 и базовым результатом (все признаки) минимальна.

Уменьшение числа признаков до 50 привело к заметному ухудшению качества. Это означает, что признаки с 51 по 100 позицию содержат важную для предсказания информацию.

LightGBM устойчив к большому количеству признаков и способен самостоятельно отсекать неинформативные за счёт встроенной регуляризации и механизма важности признаков. Искусственное удаление признаков только ухудшает результат.

### 3. Создание новых признаков

Анализ важности признаков

Выбор важных признаков проводим с помощью встроенного механизма feature_importances в LightGBM, который показывает, как часто каждый признак используется для разбиения узлов в деревьях решений. Для каждой из трёх целевых переменных (IC50, CC50, SI) обучаем модель с подобранными гиперпараметрами и получаем importance каждого признака. Затем ранжируем признаки по убыванию важности и выделяем топ-20 для каждой цели. Анализируем пересечение этих множеств — признаки, которые важны для всех трёх целей, потенциально наиболее информативны. Также выявляем признаки с нулевой важностью, которые можно исключить из дальнейшего рассмотрения.

In [27]:
best_params = {
    'IC50, mM': {'num_leaves': 31, 'min_child_samples': 20, 'learning_rate': 0.05},
    'CC50, mM': {'num_leaves': 15, 'min_child_samples': 10, 'learning_rate': 0.1},
    'SI': {'num_leaves': 31, 'min_child_samples': 20, 'learning_rate': 0.05}
}


print("\nАнализ важности признаков для каждой цели")


for target in targets:
    print(f"\nцель: {target}")

    X_train, X_val, y_train, y_val, feature_cols = split_data(df_train, target)

    model = LGBMRegressor(**best_params[target], random_state=42, verbosity=-1)
    rmse, trained_model = train_and_evaluate(
        model, X_train, X_val, y_train, y_val,
        use_log=True, use_scaling=True, scaler_type='robust'
    )
    print(f"\nrmse на валидации: {rmse:.4f}")

    importance = trained_model.feature_importances_
    importance_dict = dict(zip(feature_cols, importance))
    sorted_importance = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)

    print(f"\nтоп-20 важных признаков:")
    for i, (feat, imp) in enumerate(sorted_importance[:20], 1):
        print(f"  {i:2d}. {feat}: {imp}")

    print(f"\nПризнаки с нулевой важностью: {sum(1 for imp in importance if imp == 0)} из {len(importance)}")


print("\nАнализ общих признаков")

all_important_sets = {}
for target in targets:
    X_train, X_val, y_train, y_val, feature_cols = split_data(df_train, target)
    model = LGBMRegressor(**best_params[target], random_state=42, verbosity=-1)
    _, trained_model = train_and_evaluate(
        model, X_train, X_val, y_train, y_val,
        use_log=True, use_scaling=True, scaler_type='robust'
    )
    importance = trained_model.feature_importances_
    importance_dict = dict(zip(feature_cols, importance))
    sorted_importance = sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)
    all_important_sets[target] = set([feat for feat, imp in sorted_importance[:20]])

common_all = all_important_sets['IC50, mM'] & all_important_sets['CC50, mM'] & all_important_sets['SI']
print(f"\nпризнаки в топ-20 для всех трёх целей ({len(common_all)}):")
for feat in sorted(common_all):
    print(f"  - {feat}")



Анализ важности признаков для каждой цели

цель: IC50, mM

rmse на валидации: 397.7919

топ-20 важных признаков:
   1. FpDensityMorgan3: 69
   2. qed: 52
   3. PEOE_VSA6: 52
   4. EState_VSA8: 52
   5. MinPartialCharge: 51
   6. VSA_EState4: 51
   7. MinAbsEStateIndex: 50
   8. VSA_EState6: 48
   9. EState_VSA3: 45
  10. SPS: 43
  11. VSA_EState8: 43
  12. BCUT2D_MRLOW: 41
  13. Chi4v: 41
  14. BCUT2D_LOGPLOW: 39
  15. VSA_EState5: 39
  16. BCUT2D_CHGHI: 35
  17. EState_VSA5: 35
  18. Chi1v: 34
  19. Chi2v: 34
  20. PEOE_VSA9: 34

Признаки с нулевой важностью: 88 из 210

цель: CC50, mM

rmse на валидации: 471.8413

топ-20 важных признаков:
   1. PEOE_VSA6: 41
   2. qed: 38
   3. BCUT2D_MRLOW: 37
   4. MolLogP: 37
   5. BCUT2D_MWHI: 35
   6. BalabanJ: 35
   7. VSA_EState5: 30
   8. SPS: 29
   9. PEOE_VSA8: 29
  10. AvgIpc: 28
  11. EState_VSA4: 27
  12. Kappa1: 26
  13. BCUT2D_CHGHI: 25
  14. MinAbsEStateIndex: 24
  15. FpDensityMorgan3: 23
  16. BCUT2D_LOGPHI: 23
  17. Kappa3: 23
  18

Анализ важности признаков показал, что из 210 дескрипторов 83-88 признаков имеют нулевую важность для каждой из целевых переменных, то есть фактически не используются моделью. Удалять их не будем, так как LightGBM устойчив к шумовым признакам и способен автоматически их игнорировать.

 Выделим 6 признаков, которые входят в топ-20 для всех трёх целей одновременно: BCUT2D_MRLOW, FpDensityMorgan3, MinAbsEStateIndex, VSA_EState4, VSA_EState5, qed. Эти признаки наиболее информативны и будут использованы для создания новых комбинированных признаков (отношения, произведения, разности, суммы), которые могут выявить скрытые нелинейные зависимости и улучшить качество предсказаний.

In [28]:
best_params = {
    'IC50, mM': {'num_leaves': 31, 'min_child_samples': 20, 'learning_rate': 0.05},
    'CC50, mM': {'num_leaves': 15, 'min_child_samples': 10, 'learning_rate': 0.1},
    'SI': {'num_leaves': 31, 'min_child_samples': 20, 'learning_rate': 0.05}
}

# 6 общих важных признаков
important_features = ['BCUT2D_MRLOW', 'FpDensityMorgan3', 'MinAbsEStateIndex',
                      'VSA_EState4', 'VSA_EState5', 'qed']

def add_new_features(df, feature_list):
    """
    Добавляет новые комбинированные признаки на основе важных
    """
    df_new = df.copy()

    for i, f1 in enumerate(feature_list):
        for f2 in feature_list[i+1:]:
            if f1 in df.columns and f2 in df.columns:
                df_new[f'ratio_{f1}_{f2}'] = df[f1] / (df[f2] + 1e-8)
                df_new[f'diff_{f1}_{f2}'] = df[f1] - df[f2]
                df_new[f'mult_{f1}_{f2}'] = df[f1] * df[f2]
                df_new[f'sum_{f1}_{f2}'] = df[f1] + df[f2]

    df_new['mean_important'] = df[feature_list].mean(axis=1)
    df_new['std_important'] = df[feature_list].std(axis=1)
    df_new['min_important'] = df[feature_list].min(axis=1)
    df_new['max_important'] = df[feature_list].max(axis=1)

    return df_new

df_train_new = add_new_features(df_train, important_features)
df_test_new = add_new_features(df_test, important_features)

print(f"было признаков: {len([c for c in df_train.columns if c not in ['index', 'IC50, mM', 'CC50, mM', 'SI']])}")
print(f"стало признаков: {len([c for c in df_train_new.columns if c not in ['index', 'IC50, mM', 'CC50, mM', 'SI']])}")
print(f"добавлено новых: {len([c for c in df_train_new.columns if c not in df_train.columns])}")

results_orig = {}
results_new = {}

for target in targets:
    print(f"\n{target}")

    X_train, X_val, y_train, y_val, _ = split_data(df_train, target)
    model = LGBMRegressor(**best_params[target], random_state=42, verbosity=-1)
    rmse_orig, _ = train_and_evaluate(model, X_train, X_val, y_train, y_val, use_log=True, use_scaling=True, scaler_type='robust')
    results_orig[target] = rmse_orig

    X_train_new, X_val_new, y_train_new, y_val_new, _ = split_data(df_train_new, target)
    model_new = LGBMRegressor(**best_params[target], random_state=42, verbosity=-1)
    rmse_new, _ = train_and_evaluate(model_new, X_train_new, X_val_new, y_train_new, y_val_new, use_log=True, use_scaling=True, scaler_type='robust')
    results_new[target] = rmse_new

    print(f"  исходные: {rmse_orig:.4f} -> новые: {rmse_new:.4f}")

avg_orig = np.mean(list(results_orig.values()))
avg_new = np.mean(list(results_new.values()))

print(f"\nсредний rmse: исходные {avg_orig:.4f} -> новые {avg_new:.4f}")
print(f"изменение: {avg_new - avg_orig:+.4f}")



было признаков: 210
стало признаков: 274
добавлено новых: 64

IC50, mM
  исходные: 397.7919 -> новые: 427.6211

CC50, mM
  исходные: 471.8413 -> новые: 489.0211

SI
  исходные: 165.4725 -> новые: 169.4687

средний rmse: исходные 345.0352 -> новые 362.0370
изменение: +17.0017


Добавление новых комбинированных признаков на основе топ-20 важных дескрипторов не привело к улучшению качества модели. Средний RMSE ухудшился с 345.04 до 362.04. Для IC50 ошибка выросла на 29.83, для CC50 — на 17.18, для SI — на 4.0. Это говорит о том, что искусственное создание признаков путём линейных комбинаций (отношения, разности, произведения) не выявляет значимых нелинейных зависимостей в данных, а скорее добавляет шум, который ухудшает обобщающую способность модели. Оставляем исходный набор из 210 признаков без добавления новых.

### Эксперимент с логарифмированием целевых переменных

В ходе предыдущих экспериментов (в блокноте не отражены) было установлено, что логарифмическое преобразование целевых переменных (log1p) критически важно для получения качественных предсказаний. Без него ошибка модели вырастала более чем на 20%. Однако оставался открытым вопрос: является ли log1p оптимальным выбором, или другие виды логарифмического преобразования могут дать ещё лучшее качество.

Целевые переменные IC50, CC50 и SI имеют колоссальный разброс значений — от тысячных долей до тысяч единиц. Различные типы логарифмического преобразования по-разному сжимают этот диапазон, что может влиять на способность модели обучаться. Кроме того, существуют методы, которые автоматически подбирают оптимальный параметр преобразования под конкретное распределение данных.

Проверенные методы
1. log1p (базовый метод)
Формула: log(1 + x)
Использовался в качестве baseline. Преобразование, при котором к значению прибавляется единица перед логарифмированием, что позволяет корректно обрабатывать нулевые и близкие к нулю значения. Является стандартным выбором для задач с положительными значениями, имеющими экспоненциальное распределение.

2. log10 (десятичный логарифм)
Формула: log10(x + 1e-8)
Логарифм по основанию 10. Сжимает диапазон значений немного сильнее, чем натуральный логарифм, так как log10(1000) = 3 против ln(1000) ≈ 6.9. Может быть более удобным для интерпретации, так как изменение на 1 по шкале log10 соответствует изменению в 10 раз исходного значения.

3. log2 (двоичный логарифм)
Формула: log2(x + 1e-8)
Логарифм по основанию 2. Сжимает диапазон значений сильнее всех: log2(1000) ≈ 10. Изменение на 1 по шкале log2 соответствует изменению в 2 раза исходного значения. Потенциально может лучше подходить для данных, где значимые изменения происходят в геометрической прогрессии со знаменателем 2.

4. Box-Cox
Формула: (x^λ - 1) / λ при λ ≠ 0, ln(x) при λ = 0
Метод, который автоматически подбирает оптимальный параметр λ, максимизирующий нормальность распределения преобразованных данных. Требует строго положительных значений (добавляется небольшой сдвиг 1e-8). Теоретически может подобрать наиболее подходящее преобразование для каждой целевой переменной индивидуально, так как λ вычисляется отдельно для IC50, CC50 и SI.

5. Yeo-Johnson
Метод, аналогичный Box-Cox, но не требующий строгой положительности данных. Работает с любыми значениями, включая отрицательные и нулевые. Как и Box-Cox, автоматически подбирает оптимальный параметр преобразования. Более универсален, но может быть избыточен для данной задачи, так как наши целевые переменные всегда положительны.

### Далее эксперименты проводим с обучением на всех данных и загрузкой результатов на Kaggle

In [35]:
# Новые функции для альтернативных преобразований

def log_transform(y):
    return np.log1p(y)

def log10_transform(y):
    return np.log10(y + 1e-8)

def log2_transform(y):
    return np.log2(y + 1e-8)

def boxcox_transform(y):
    # Box-Cox требует положительных значений, добавляем сдвиг
    y_shifted = y + 1e-8
    transformed, lambda_ = boxcox(y_shifted)
    return transformed, lambda_

def yeojohnson_transform(y):
    pt = PowerTransformer(method='yeo-johnson')
    transformed = pt.fit_transform(y.values.reshape(-1, 1)).flatten()
    return transformed, pt

def exp_back_transform(y_transformed, method='log1p', lambda_=None, pt=None):
    if method == 'log1p':
        return np.expm1(y_transformed)
    elif method == 'log10':
        return 10 ** y_transformed
    elif method == 'log2':
        return 2 ** y_transformed
    elif method == 'boxcox':
        return inv_boxcox(y_transformed, lambda_)
    elif method == 'yeojohnson':
        return pt.inverse_transform(y_transformed.reshape(-1, 1)).flatten()

# Модифицируем train_final_and_predict для разных преобразований
def train_final_and_predict_custom(model, df_train, df_test, target_col,
                                    transform_method='log1p', use_scaling=True):
    feature_cols = [col for col in df_train.columns
                    if col not in ['index', 'IC50, mM', 'CC50, mM', 'SI']]

    X_full = df_train[feature_cols].copy()
    y_full = df_train[target_col].copy()
    X_test = df_test[feature_cols].copy()

    # Применяем выбранное преобразование
    if transform_method == 'log1p':
        y_full_transformed = np.log1p(y_full)
        back_transform = lambda x: np.expm1(x)
    elif transform_method == 'log10':
        y_full_transformed = np.log10(y_full + 1e-8)
        back_transform = lambda x: 10 ** x
    elif transform_method == 'log2':
        y_full_transformed = np.log2(y_full + 1e-8)
        back_transform = lambda x: 2 ** x
    elif transform_method == 'boxcox':
        y_shifted = y_full + 1e-8
        y_full_transformed, lambda_ = boxcox(y_shifted)
        back_transform = lambda x: inv_boxcox(x, lambda_)
    elif transform_method == 'yeojohnson':
        pt = PowerTransformer(method='yeo-johnson')
        y_full_transformed = pt.fit_transform(y_full.values.reshape(-1, 1)).flatten()
        back_transform = lambda x: pt.inverse_transform(x.reshape(-1, 1)).flatten()

    if use_scaling:
        scaler = RobustScaler()
        X_full_scaled = scaler.fit_transform(X_full)
        X_test_scaled = scaler.transform(X_test)
        X_full_processed = X_full_scaled
        X_test_processed = X_test_scaled
    else:
        X_full_processed = X_full
        X_test_processed = X_test

    model.fit(X_full_processed, y_full_transformed)
    pred_test_transformed = model.predict(X_test_processed)
    predictions = back_transform(pred_test_transformed)

    return predictions

from scipy.special import inv_boxcox

# Лучшие параметры
best_params = {
    'IC50, mM': {'num_leaves': 31, 'min_child_samples': 20, 'learning_rate': 0.05},
    'CC50, mM': {'num_leaves': 15, 'min_child_samples': 10, 'learning_rate': 0.1},
    'SI': {'num_leaves': 31, 'min_child_samples': 20, 'learning_rate': 0.05}
}

targets = ['IC50, mM', 'CC50, mM', 'SI']

# Тестируем разные преобразования
transform_methods = ['log1p', 'log10', 'log2', 'boxcox', 'yeojohnson']

print("Разные типы логарифмирования")

for method in transform_methods:
    print(f"Метод: {method}")

    predictions = {}

    for target in targets:
        print(f"  {target}")
        model = LGBMRegressor(**best_params[target], random_state=42, verbosity=-1)

        try:
            preds = train_final_and_predict_custom(model, df_train, df_test, target,
                                                    transform_method=method, use_scaling=True)
            predictions[target] = preds
            print(f"успешно")
        except Exception as e:
            print(f"ошибка: {str(e)[:50]}")
            continue

    if len(predictions) == 3:
        submission = pd.DataFrame({
            'index': df_sample['index'],
            'IC50': predictions['IC50, mM'],
            'CC50': predictions['CC50, mM'],
            'SI': predictions['SI']
        })
        submission.to_csv(f'submission_{method}.csv', index=False)
        print(f"\nохранён submission_{method}.csv")

print("1. submission_log1p.csv")
print("2. submission_log10.csv - десятичный логарифм")
print("3. submission_log2.csv - двоичный логарифм")
print("4. submission_boxcox.csv - Box-Cox преобразование")
print("5. submission_yeojohnson.csv - Yeo-Johnson преобразование")

Разные типы логарифмирования
Метод: log1p
  IC50, mM
успешно
  CC50, mM
успешно
  SI
успешно

охранён submission_log1p.csv
Метод: log10
  IC50, mM
успешно
  CC50, mM
успешно
  SI
успешно

охранён submission_log10.csv
Метод: log2
  IC50, mM
успешно
  CC50, mM
успешно
  SI
успешно

охранён submission_log2.csv
Метод: boxcox
  IC50, mM
успешно
  CC50, mM
успешно
  SI
успешно

охранён submission_boxcox.csv
Метод: yeojohnson
  IC50, mM
успешно
  CC50, mM
успешно
  SI
успешно

охранён submission_yeojohnson.csv
1. submission_log1p.csv
2. submission_log10.csv - десятичный логарифм
3. submission_log2.csv - двоичный логарифм
4. submission_boxcox.csv - Box-Cox преобразование
5. submission_yeojohnson.csv - Yeo-Johnson преобразование


Лучший результат на Kaggle показал Box-Cox с результатом 295.85, что на 7.65 пункта лучше, чем baseline с log1p (303.50).

Box-Cox превзошёл другие методы благодаря тому, что автоматически подбирает оптимальный параметр λ для каждого таргета индивидуально. Для IC50, CC50 и SI распределения отличаются, и единый подход (log1p, log10 или log2) не может быть оптимальным для всех трёх целей одновременно. Box-Cox решил эту проблему, найдя наилучшее преобразование для каждой целевой переменной отдельно